# 5.1. Multilayer Perceptrons
D2L의 Multilayer Perceptrons장을 PyTorch 기준으로 정리함.

- 선형 모델이 복잡한 관계를 표현하기 어려운 이유
- 은닉층의 역할
- 층을 여러 개 쌓기만 해서는 표현력이 증가하지 않는 이유
- 비선형 활성화 함수가 필요한 이유
- ReLU, Sigmoid, Tanh 함수의 특징과 미분
- MLP가 복잡한 함수를 근사할 수 있는 이유

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [2]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 선형 모델에서 신경망으로

앞에서는 softmax regression을 사용해서 Fashion-MNIST이미지를 분류했다.

Softmax regression은 입력을 출력에 직접 연결하는 선형 모델이다.

$$
\mathbf{O} = \mathbf{XW} + \mathbf{b}
$$

이런 선형 모델은 입력과 출력 사이의 관계가 단순한 경우에 효과적이다.

이미지, 음성, 자연어처럼 복잡한 데이터에는 입력 특성 사이의 상호작용이 존재한다. 그래서 입력과 출력 사이의 관계를 하나의 선형 변환만으로 표현하기 어렵다.

## 2. 선형 모델의 한계

선형 모델에서는 하나의 입력 특성이 증가할 때 출력이 항상 같은 방향으로 변한다.

가중치가 양수라면 입력이 증가할수록 출력도 증가하고, 가중치가 음수라면 입력이 증가할수록 출력은 감소한다.

선형모델은 기본적으로 다음과 같은 관계를 가정한다.

$$
y = wx + b
$$

하지만 현실의 데이터는 이러한 관계를 따르지 않는 경우가 많다.

예를 들어서 체온과 건강 위험도의 관계를 생각해보자.

체온이 정상 범위보다 높아지거나 낮아지면 위험도가 증가한다.

따라서 체온과 위험도의 관계는 단순히 계속 증가하거나 계속 감소하는 선형 관계가 아니다.

이미지 데이터에서는 문제가 더 복잡하다.

특정 픽셀 하나의 밝기만으로 고양이와 강아지를 구분할 수 없다. 특정 픽셀의 의미는 주변 픽셀, 윤곽선, 모양 등과 함께 결정된다.

따라서 신경망은 단순히 입력을 출력으로 변환하는 것뿐만 아니라, 입력으로부터 유용한 표현을 함께 학습해야 한다.

## 3. 은닉층

선형 모델보다 복잡한 관계를 학습하기 위해 입력층과 출력층 사이에 새로운 층을 추가할 수 있다. 이 층을 은닉층(hidden layer)이라고 한다.

$$
\text{입력층}
\rightarrow
\text{은닉층}
\rightarrow
\text{출력층}
$$

은닉층은 입력데이터를 그대로 출력하지 않고 입력 특성을 조합해 모델이 문제를 해결하는데 필요한 새로운 표현을 만든다.

여러 개의 완전연결층을 쌓아 만든 신경망을 다층 퍼셉트론이라고 한다.

예를 들어 Fashion-MNIST 이미지가 입력으로 들어오면 은닉층은 다음과 같은 패턴을 학습할 수 있다.

- 선의 방향
- 옷의 경계
- 밝고 어두운 영역

모델은 학습 과정에서 손실을 줄이는 방향으로 유용한 특성 표현을 스스로 만든다.

## 4. 은닉층의 계산

미니배치 크기를 $n$, 입력 특성 수를 $d$, 은닉 유닛 수를 $h$라고 하자. 

입력 데이터의 크기는 다음과 같다. 

$$ 
\mathbf{X} \in \mathbb{R}^{n \times d} 
$$ 

입력층과 은닉층 사이의 가중치와 편향은 다음과 같다. 

$$ 
\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h} 
$$
$$ 
\mathbf{b}^{(1)} \in \mathbb{R}^{h} 
$$ 
 
은닉층의 계산 결과는 다음과 같다. 

$$ 
\mathbf{H} = \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)} 
$$ 

은닉층 출력의 크기는 다음과 같다. 

$$ 
\mathbf{H} \in \mathbb{R}^{n \times h} 
$$ 

각 샘플은 기존의 $d$개 입력 특성에서 새로운 $h$개 은닉 특성으로 변환된다.

In [ ]:
batch_size = 4 
input_dim = 784 
hidden_dim = 256 

X = torch.randn(batch_size, input_dim) 
W1 = torch.randn(input_dim, hidden_dim) 
b1 = torch.randn(hidden_dim) 

H = X @ W1 + b1

print("X shape:", X.shape) 
print("W1 shape:", W1.shape) 
print("b1 shape:", b1.shape) # 편향
print("H shape:", H.shape) # 은닉층

X shape: torch.Size([4, 784])
W1 shape: torch.Size([784, 256])
b1 shape: torch.Size([256])
H shape: torch.Size([4, 256])


## 5. 완전 연결층

은닉층의 각 뉴런이 이전 층의 모든 입력과 연결된 층을 완전 연결층(fully connected layer)이라고 한다.

예를 들어서 입력 특성이 4이고 은닉 유닛이 3개라면 각 은닉 유닛은 4개의 입력을 모두 사용한다.

각 은닉 유닛은 다음과 같은 계산을 수행한다.
$$
\sum_{i=1}^{d} x_i w_{ij}+b_j
$$

여기서 $h_j$는 $j$번째 은닉 유닛의 출력값을 의미한다.

PyTorch에서는 완전 연결층을 nn.Linear로 구현한다.

    nn.Linear(input_features, output_features)

예를 들어 다음 층은 784개의 입력 특성을 256개의 은닉 특성으로 변환한다.

    nn.Linear(784, 256)

입력과 출력의 크기는 다음과 같다.

입력 : [batch_size, 784]
출력 : [batch_size, 256]

In [4]:
hidden_layer = nn.Linear(784, 256) 

X = torch.randn(32, 784) 
H = hidden_layer(X) 

print("입력 shape:", X.shape) 
print("출력 shape:", H.shape)

입력 shape: torch.Size([32, 784])
출력 shape: torch.Size([32, 256])


완전 연결층의 `nn.Linear(784, 256)`은 내부적으로 다음 파라미터를 가진다.

- 가중치
- 편향

다만 PyTorch의 가중치 저장 형태는 수학식에서 사용한 행렬 방향과 다르다.

수학식에서는 다음과 같이 표현했다.

$$
\mathbf{X}\mathbf{W}
$$

이때 가중치의 크기는 다음과 같다.

$$
\mathbf{W} \in \mathbb{R}^{784 \times 256}
$$

하지만 `nn.Linear(784, 256)` 내부의 가중치 shape은 다음과 같다.

```text
[256, 784]
```
PyTorch가 내부적으로 다음 연산을 수행하기 때문이다.

$$
\mathbf{X}\mathbf{W}^{T}+\mathbf{b}
$$

따라서 nn.Linear의 가중치 shape은 다음 순서로 저장된다.

[out_features, in_features]

In [5]:
layer = nn.Linear(784, 256) 

print("weight shape:", layer.weight.shape) 
print("bias shape:", layer.bias.shape)

weight shape: torch.Size([256, 784])
bias shape: torch.Size([256])


출력 클래스가 10개라면 은닉층에서 출력층으로 가는 두 번째 완전 연결층을 추가할 수 있다. 

$$ 
\mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)} 
$$ 
두 번째 층의 파라미터 크기는 다음과 같다. 
$$ 
\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q} 
$$
$$ 
\mathbf{b}^{(2)} \in \mathbb{R}^{q} 
$$

만약에 은닉 유닛이 256개이고 출력 클래스가 10개면

H : [batch_size, 256] 
W2 : [256, 10] 
b2 : [10] 
O : [batch_size, 10]

입력 [batch_size, 784]  
↓ Linear(784, 256)   
은닉층 [batch_size, 256]   
↓ Linear(256, 10)   
출력 [batch_size, 10]  

이렇다.

In [ ]:
batch_size = 32 
input_dim = 784 
hidden_dim = 256 
output_dim = 10 

X = torch.randn(batch_size, input_dim) 

layer1 = nn.Linear(input_dim, hidden_dim) 
layer2 = nn.Linear(hidden_dim, output_dim)

H = layer1(X) 
O = layer2(H) 

print("X shape:", X.shape) # 입력
print("H shape:", H.shape) # 은닉층
print("O shape:", O.shape) # 출력

X shape: torch.Size([32, 784])
H shape: torch.Size([32, 256])
O shape: torch.Size([32, 10])


## 6. 활성화 함수가 필요한 이유

현재 신경망은 두개의 완전 연결층을 가지고 있다.

$$ 
\mathbf{H} = \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)} 
$$ 
$$ 
\mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)} 
$$

첫 번째 식을 두 번째 식에 대입하면 다음과 같다.

$$
\mathbf{O}
=
\left(
\mathbf{X}\mathbf{W}^{(1)}
+
\mathbf{b}^{(1)}
\right)
\mathbf{W}^{(2)}
+
\mathbf{b}^{(2)}
$$

식을 전개하면 다음과 같다.

$$
\mathbf{O}
=
\mathbf{X}\mathbf{W}^{(1)}\mathbf{W}^{(2)}
+
\mathbf{b}^{(1)}\mathbf{W}^{(2)}
+
\mathbf{b}^{(2)}
$$

새로운 가중치와 편향을 다음처럼 정의할 수 있다.

$$
\mathbf{W}
=
\mathbf{W}^{(1)}\mathbf{W}^{(2)}
$$

$$
\mathbf{b}
=
\mathbf{b}^{(1)}\mathbf{W}^{(2)}
+
\mathbf{b}^{(2)}
$$

그러면 전체 식은 다시 다음처럼 바뀐다.

$$
\mathbf{O}
=
\mathbf{X}\mathbf{W}
+
\mathbf{b}
$$

선형층을 두 개 연결했지만 결국 하나의 선형층과 동일해졌다.

```text
Linear → Linear

은 결국 다음과 같다.

Linear

선형층을 여러 개 쌓는 것만으로는 복잡한 비선형 관계를 표현할 수 없다는 뜻이다.

이런 문제를 해결하기 위해서 은닉층의 선형 변환 결과에 활성화 함수를 적용한다.

활성화 함수를 $\sigma$라고 하면 은닉층은 다음과 같이 계산한다.

$$ 
\mathbf{H} = \sigma \left( \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)} \right) 
$$ 
출력층은 다음과 같다.

$$
\mathbf{O}=\mathbf{H}\mathbf{W}^{(2)}+\mathbf{b}^{(2)}
$$

전체 구조는 이렇다.

```text
입력
  ↓
선형 변환
  ↓
활성화 함수
  ↓
선형 변환
  ↓
출력
```
활성화 함수는 선형 변환 결과를 비선형적으로 바꾼다.

따라서 두 개의 선형층을 하나의 선형층으로 합칠 수 없게 된다.

이런 비선형성 덕분에 MLP는 단순한 직선보다 복잡한 결정 경계를 학습할 수 있다고 한다.

In [7]:
X = torch.randn(32, 784) 

layer1 = nn.Linear(784, 256) 
activation = nn.ReLU() 
layer2 = nn.Linear(256, 10)

H_linear = layer1(X) 
H = activation(H_linear) 
O = layer2(H) 

print("선형 변환 후:", H_linear.shape) 
print("활성화 함수 후:", H.shape) 
print("최종 출력:", O.shape)

선형 변환 후: torch.Size([32, 256])
활성화 함수 후: torch.Size([32, 256])
최종 출력: torch.Size([32, 10])


## 7. ReLU

ReLU는 현재 가장 널리 사용되는 활성화 함수 중 하나이다.

ReLU는 이렇게 정의한다

$$ 
\operatorname{ReLU}(x) = \max(x, 0)
$$

입력이 양수면 그대로 출력하고, 입력이 0이하면 0을 출력한다.

텐서의 shape을 변경하지 않는다.

입력값의 일부만 0으로 바꾸게 한다.

In [8]:
x = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0]) 

y = torch.relu(x) 

print("입력:", x) 
print("출력:", y)

입력: tensor([-3., -1.,  0.,  2.,  5.])
출력: tensor([0., 0., 0., 2., 5.])


ReLU 미분은 이렇게 생각할 수 있다.

$$ 
\frac{d}{dx}\operatorname{ReLU}(x) = \begin{cases} 0, & x \leq 0 \\ 1, & x > 0 \end{cases} 
$$

- 입력이 음수면 gradient 0
- 입력이 양수면 gradient 1

ReLU는 양수 영역에서 gradient가 줄지 않고 그대로 전달된다.

그래서 Sigmoid나 Tanh보다 깊은 신경망에서 학습하기 유리한 경우가 많다고 한다.

하지만 어떤 뉴런의 입력이 계속 음수면 이런 문제가 발생한다

출력 0 , gradient = 0

이 뉴런은 더 이상 제대로 학습되지 않을 수 있다.

이를 Dying ReLU문제 라고 한다고 한다.

## 8. Sigmoid, Tanh

### Sigmoid

Sigmoid 함수는 다음과 같이 정의한다.

$$ 
\operatorname{sigmoid}(x) = \frac{1}{1+e^{-x}} 
$$ 

출력값은 항상 0과 1 사이이다. 

$$ 
0 < \operatorname{sigmoid}(x) < 1 
$$ 

대표적인 출력값은 이렇다.

```text
매우 작은 음수 -> 0에 가깝다
0 -> 0.5
매우 큰 양수 -> 1에 가깝다
```
Sigmoid는 출력값을 확률처럼 해석하기 쉽다. 그래서 이진 분류의 출력과 정보의 통과 비율을 결정하는 게이트 구조에서 사용된다고 한다.

In [9]:
x = torch.tensor([-5.0, 0.0, 5.0]) 

y = torch.sigmoid(x) 

print(y)

tensor([0.0067, 0.5000, 0.9933])


Sigmoid의 미분은 다음과 같다.

$$
\frac{d}{dx}\operatorname{sigmoid}(x)
=
\operatorname{sigmoid}(x)
\left(
1-\operatorname{sigmoid}(x)
\right)
$$

입력이 0일 때 gradient가 가장 크다.

$$
\operatorname{sigmoid}(0) = 0.5
$$

$$
0.5(1-0.5)=0.25
$$

Sigmoid의 최대 gradient는 0.25이다.

입력의 절댓값이 커질수록 gradient가 0에 가까워진다.

```text
큰 음수 → gradient가 거의 0
0 근처  → gradient가 비교적 큼
큰 양수 → gradient가 거의 0
```

신경망이 깊어지면 작은 gradient가 여러 층을 거치며 계속 곱해질 수 있다.

그러면 앞쪽 층의 gradient가 거의 0이 되는 기울기 소실(vanishing gradient)이 발생할 수 있다.

따라서 일반적인 MLP 은닉층에서는 Sigmoid보다 ReLU를 더 자주 사용한다고 한다.

### Tanh

Tanh는 하이퍼볼릭 탄젠트 함수이다.

$$
\operatorname{tanh}(x)=\frac{1-e^{-2x}}{1+e^{-2x}}
$$

출력 범위는 -1과 1 사이이다.

$$
-1<\operatorname{tanh}(x)<1
$$

Tanh는 Sigmoid와 그래프가 비슷한 S자 형태이지만 출력의 중심이 0이다.

```text
Sigmoid 출력 범위:  0 ~ 1
Tanh 출력 범위:    -1 ~ 1
```
Tanh의 미분은 다음과 같다.
$$
1-\operatorname{tanh}^{2}(x)
$$

입력이 0일 때 gradient는 1이다. (1 - 0^2 = 1)

하지만 입력의 절댓값이 커질수록 gradient가 0에 가까워지므로 Tanh 역시 기울기 소실이 발생할 수 있다.

In [10]:
x = torch.tensor([-5.0, 0.0, 5.0]) 

sigmoid_output = torch.sigmoid(x) 
tanh_output = torch.tanh(x) 

print("Sigmoid:", sigmoid_output) 
print("Tanh:", tanh_output)

Sigmoid: tensor([0.0067, 0.5000, 0.9933])
Tanh: tensor([-0.9999,  0.0000,  0.9999])


## 9. MLP 구조 확인

Fashion-MNIST 이미지 한장의 크기는 [1, 28, 28] 이다.

`nn.Flatten()`을 적용하면 채널, 높이, 너비 차원이 하나로 합쳐진다 (784)

그래서 shape가 이렇게 된다 [batch_size, 784]

In [11]:
model = nn.Sequential( 
    nn.Flatten(), 
    nn.Linear(28 * 28, 256), 
    nn.ReLU(), 
    nn.Linear(256, 10)
) 

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)


shape 흐름은 이렇게 된다.

```text
입력 이미지 [batch_size, 1, 28, 28] 

↓ nn.Flatten() 

펼친 이미지 [batch_size, 784] 

↓ nn.Linear(784, 256) 

은닉층 선형 출력 [batch_size, 256] 

↓ nn.ReLU() 

은닉층 활성값 [batch_size, 256] 

↓ nn.Linear(256, 10) 

클래스별 출력 점수 [batch_size, 10]
```

마지막 nn.Linear(256, 10)의 출력에는 10개 클래스에 대한 점수가 들어있다. (logit)

In [12]:
X = torch.randn(32, 1, 28, 28) 

output = model(X) 

print("입력 shape:", X.shape) 
print("출력 shape:", output.shape)

입력 shape: torch.Size([32, 1, 28, 28])
출력 shape: torch.Size([32, 10])


다중 클래스 분류에서 마지막 층 뒤에 `nn.Softmax()`를 직접 추가하지 않는 경우가 많다고 한다.

PyTorch의 `nn.CrossEntropyLoss()`가 내부적으로 logits에 필요한 연산을 처리하기 때문이다. 

일반적인 구조는 다음과 같다.

```text
모델 마지막 층 
→ nn.Linear(256, 10) 

손실 함수 
→ nn.CrossEntropyLoss()
```

모델은 Softmax가 적용되지 않은 원래 점수 logit값을 출력한다.

    loss_fn = nn.CrossEntropyLoss() 
    loss = loss_fn(output, labels)

## 10. 오늘의 정리

- 선형 모델은 입력과 출력 사이의 복잡한 비선형 관계를 표현하는 데 한계가 있다. 
- 입력층과 출력층 사이에 있는 층을 은닉층(hidden layer)이라고 한다. 
- 은닉층은 기존 입력 특성을 조합하여 새로운 특징 표현을 학습한다. 
- 이전 층의 모든 입력과 연결된 층을 완전 연결층(fully connected layer)이라고 하며, PyTorch에서는 `nn.Linear`로 구현한다. 
- `nn.Linear(784, 256)`은 784개의 입력 특성을 256개의 새로운 특성으로 변환한다. 
- 선형층을 여러 개 연속으로 쌓아도 결국 하나의 선형 변환으로 합칠 수 있다. 
- 따라서 신경망이 복잡한 관계를 학습하려면 비선형 활성화 함수(activation function)가 필요하다. 
- ReLU는 음수를 0으로 만들고 양수는 그대로 전달하며, 일반적인 MLP 은닉층에서 많이 사용한다. 
- Sigmoid는 출력 범위가 `0~1`, Tanh는 `-1~1`이며 입력의 절댓값이 커지면 gradient가 작아져 기울기 소실이 발생할 수 있다. 
- `nn.ReLU()` 같은 활성화 함수는 값은 바꾸지만 tensor의 shape은 바꾸지 않는다. 
- Fashion-MNIST에서 `nn.Flatten()` 후 784가 되는 이유는 이미지 크기가 `1 × 28 × 28`이기 때문이다. 
- MLP의 기본 흐름은 `Flatten → Linear → ReLU → Linear`로 이해하면 된다. 
- 다중 클래스 분류에서는 마지막 `Linear`의 출력인 logits를 `nn.CrossEntropyLoss()`에 바로 전달할 수 있다.